# Logical Expert System - KCropAI

### Logic Programming Integration
This notebook demonstrates **deterministic logic programming**. We implement a Rule-Based Expert System for Facts, Predicates, Rules, and Unification.

**Engineering Approach:**
To ensure cross-platform execution, this notebook implements a dual-engine architecture:
1. **Method A (PySwip Bridge):** Attempts to interface with the native SWI-Prolog binary via `prolog/crop_knowledge_base.pl`.
2. **Method B (Native Python Logic):** A self-contained fallback inference engine that executes if SWI-Prolog is missing from the local environment.

### Method A: PySwip (SWI-Prolog Bridge)
This section attempts to load the `.pl` file directly.

In [1]:
import os
import sys
try:
    from pyswip import Prolog
    
    # Initialize engine and load knowledge base
    prolog = Prolog()
    kb_path = os.path.abspath("../prolog/crop_knowledge_base.pl").replace("\\", "/")
    prolog.consult(kb_path)
    
    # Query Rules
    print("\nQuery: recommend_crop(laterite, high, Crop)")
    recommendations = list(prolog.query("recommend_crop(laterite, high, Crop)"))
    
    if recommendations:
        for r in recommendations:
            print(f"Recommended: {r['Crop'].title()}")
            
    # Query Lists
    print("\nQuery: crops_for_soil(alluvial, List)")
    list_results = list(prolog.query("crops_for_soil(alluvial, List)"))
    for r in list_results:
        print(f"Alluvial Crops List: {[c.title() for c in r['List']]}")

except Exception as e:
    print(f"PySwip/SWI-Prolog Initialization Failed: {type(e).__name__}")
    print("Executing fallback to Native Python Logic Engine...")


Query: recommend_crop(laterite, high, Crop)
Recommended: Coconut
Recommended: Rubber
Recommended: Black_Pepper

Query: crops_for_soil(alluvial, List)
Alluvial Crops List: ['Rice', 'Banana']


### Method B: Native Python Inference Engine (Fallback)
If standard Prolog binaries are unavailable, this class replicates the Unification and Rule-based logic using pure Python dictionaries and functions.

In [2]:
class KCropExpertSystem:
    def __init__(self):
        # 1. FACTS & PREDICATES
        self.soil_facts = {
            "coconut": "laterite", "rubber": "laterite", "black_pepper": "laterite",
            "rice": "alluvial", "banana": "alluvial", "tapioca": "red", "coffee": "laterite"
        }
        self.rain_facts = {
            "coconut": "high", "rubber": "high", "black_pepper": "high",
            "rice": "high", "banana": "moderate", "tapioca": "moderate", "coffee": "moderate"
        }

    # 2. INFERENCE RULES
    def recommend_crop(self, soil, rain):
        """Logic: recommend(C) :- soil_type(C, S), rainfall_req(C, R)."""
        results = []
        for crop in self.soil_facts:
            # Unification step
            match_soil = self.soil_facts[crop] == soil
            match_rain = self.rain_facts.get(crop) == rain
            
            if match_soil and match_rain:
                results.append(crop)
        return results
        
    def crops_for_soil(self, soil):
        """Logic: findall(C, soil_type(C, S), List)."""
        return [crop for crop, s in self.soil_facts.items() if s == soil]

# Initialize and Query
expert = KCropExpertSystem()

print("\nQuery: recommend_crop(laterite, high)")
recommendations = expert.recommend_crop("laterite", "high")
for r in recommendations:
    print(f"Recommended: {r.title()}")

print("\nQuery: crops_for_soil(alluvial)")
alluvial_list = expert.crops_for_soil("alluvial")
print(f"Alluvial Crops List: {[c.title() for c in alluvial_list]}")


Query: recommend_crop(laterite, high)
Recommended: Coconut
Recommended: Rubber
Recommended: Black_Pepper

Query: crops_for_soil(alluvial)
Alluvial Crops List: ['Rice', 'Banana']
